### Wiener's Attack on RSA

In [17]:
import random

def is_probable_prime(n, rounds = 16):
    """ Miller-Rabin primality test (probabilistic) """
    if n < 2:
        return False
    if n == 2 or n == 3:
        return True
    if n % 2 == 0:
        return False
    
    # First 100 primes for trial division
    small_primes = [
        2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71,
        73, 79, 83, 89, 97, 101, 103, 107, 109, 113, 127, 131, 137, 139, 149, 151,
        157, 163, 167, 173, 179, 181, 191, 193, 197, 199, 211, 223, 227, 229, 233,
        239, 241, 251, 257, 263, 269, 271, 277, 281, 283, 293, 307, 311, 313, 317,
        331, 337, 347, 349, 353, 359, 367, 373, 379, 383, 389, 397, 401, 409, 419,
        421, 431, 433, 439, 443, 449, 457, 461, 463, 467, 479, 487, 491, 499, 503,
        509, 521, 523, 541
    ]

    for p in small_primes:
        if n == p:
            return True
        if n % p == 0:
            return False
    
    # write n - 1 as 2^r * d
    r, d = 0, n - 1
    while d % 2 == 0:
        r += 1
        d //= 2

    for _ in range(rounds):
        a = random.randrange(2, n - 1)
        x = pow(a, d, n)

        if x == 1 or x == n - 1:
            continue

        for _ in range(r - 1):
            x = pow(x, 2, n)
            if x == n - 1:
                break

        else:
            return False
    
    return True

def generate_random_prime(bits):
    while True:
        x = random.getrandbits(bits)
        x |= (1 << bits - 1) # for top bit
        x |= 1 # for x to be odd

        if is_probable_prime(x):
            return x

In [18]:
def gcd(a, b):
    while b != 0:
        a, b = b, a % b
    return a

def extended_gcd(a, b):
    """ Returns (gcd, x, y) such that a*x + b*y = gcd(a, b) """
    if b == 0:
        return a, 1, 0
    
    gcd_val, x1, y1 = extended_gcd(b, a % b)
    x = y1
    y = x1 - (a // b) * y1
    
    return gcd_val, x, y

In [19]:
def mod_inverse(a, m):
    """ Compute modular inverse of a modulo m """
    return pow(a, -1, m)

In [20]:
from math import isqrt

def weak_rsa(bits):
  """
  Generate an RSA key vulnerable to Wiener's attack by forcing d to be small:
  d < (1/3) * n^(1/4)
  Returns (n, e, d, p, q).
  """

  while True:
    p = generate_random_prime(bits // 2)
    q = generate_random_prime(bits // 2)
    
    if p == q:
      continue

    n = p * q
    phi = (p - 1) * (q - 1)

    # We choose d such that: d < (1/3) * n^(1/4)
    # We'll pick a small d under that bound and coprime to phi.
    bound = isqrt(isqrt(n)) // 3
    if bound <= 2:
      continue

    for _ in range(10000):
      d = random.randrange(2, bound)
      if gcd(d, phi) == 1:
        e = mod_inverse(d, phi)
        if 1 < e < n:
          return n, e, d, p, q



In [21]:
n, e, d_true, p , q = weak_rsa(256)
print("Public key (n, e):")
print("n =", n)
print("e =", e)

print("\nPrivate key (n, d):")
print("n =", n)
print("d =", d_true)

# Encrypting/Decrypting a small message to see if RSA works
m = 123456789
c = pow(m, e, n)
m_decrypted = pow(c, d_true, n)
print("\nMessage test:")
print(f"m = {m}")
print(f"m encrypted = {c}")
print(f"m decrypted = {m_decrypted}")

Public key (n, e):
n = 84302377372395525654826222778953874378869265319561304797226118453962341292273
e = 52763416503189039003293699678631613392491783172996734972457688772380982675767

Private key (n, d):
n = 84302377372395525654826222778953874378869265319561304797226118453962341292273
d = 5118494515309596031

Message test:
m = 123456789
m encrypted = 4625900907607527015864836555686616524718076631786609568975245173577531938719
m decrypted = 123456789


In [7]:
def continued_fraction(numerator, denominator):
    """
    Compute the continued fraction representation of numerator/denominator.
    Returns list of quotients [a0, a1, a2, ...]
    """
    cf = []
    
    while denominator != 0:
        quotient = numerator // denominator
        cf.append(quotient)
        numerator, denominator = denominator, numerator - quotient * denominator
    
    return cf

def convergents_from_cf(cf):
    """
    Compute convergents from continued fraction representation.
    Returns list of (numerator, denominator) tuples.
    """
    convergents = []
    
    for i in range(len(cf)):
        if i == 0:
            num, den = cf[0], 1
        elif i == 1:
            num = cf[1] * cf[0] + 1
            den = cf[1]
        else:
            num = cf[i] * convergents[i-1][0] + convergents[i-2][0]
            den = cf[i] * convergents[i-1][1] + convergents[i-2][1]
        
        convergents.append((num, den))
    
    return convergents

In [8]:
def is_square(x):
    if x < 0:
        return False
    r = isqrt(x)
    if r * r == x:
        return True
    return False

In [9]:
def wiener_attack(e, n):
    """
    Attempt to recover RSA private exponent d from public (e, n)
    using Wiener's attack. Returns d if found, else None.
    """
    cf = continued_fraction(e, n)
    convergents = convergents_from_cf(cf)
    for k, d in convergents:
        if k == 0 or d == 0:
            continue
        
        # from e * d - 1 = k * phi => phi = (e * d - 1) / k
        ed_minus1 = e * d - 1
        if ed_minus1 % k != 0:
            continue
        phi = ed_minus1 // k

        # phi(n) = (p - 1)(q - 1) = n - (p + q) + 1 => p + q = n - phi + 1
        s = n - phi + 1

        # Solve x^2 - s * x + n = 0
        # Discriminant must be perfect square
        disc = s * s - 4 * n
        if not is_square(disc):
            continue

        t = isqrt(disc)
        p = (s + t) // 2
        q = (s - t) // 2
        if p > 1 and q > 1 and p * q == n:
            return d
        
    return None

In [10]:
d_recovered = wiener_attack(e, n)
print("\nd recovered =", d_recovered)
print("Success?", d_recovered == d_true)


d recovered = 732863523964224151
Success? True


### Lenstra's attack on RSA

In [22]:
def generate_rsa_key(bits):
    p = generate_random_prime(bits // 2)
    q = generate_random_prime(bits // 2)
    while q == p:
        q = generate_random_prime(bits // 2)

    n = p * q
    phi = (p - 1) * (q - 1)

    e = 65537
    if gcd(e, phi) != 1:
        e = 3
        while gcd(e, phi) != 1:
            e += 2

    d = pow(e, -1, phi)

    dp = d % (p - 1)
    dq = d % (q - 1)
    qinv = pow(q, -1, p)

    return (n, e, d, p, q, dp, dq, qinv)

In [23]:
import secrets

def rsa_crt_sign(m, p, q, dp, dq, qinv, fault_in):
    """
    Compute s = m^d mod n using CRT.
    If fault_in is "p" or "q", corrupt that branch to simulate a hardware fault.
    """
    sp = pow(m, dp, p)
    sq = pow(m, dq, q)

    if fault_in == "p":
        # p is the corrupted one
        sp = (sp + secrets.randbelow(p - 1) + 1) % p
    elif fault_in == "q":
        # q is the corrupted one
        sq = (sq + secrets.randbelow(q - 1) + 1) % q

    h = (qinv * (sp - sq)) % p
    s = sq + q * h
    return s

In [24]:
def lenstra_recover_with_public(n, e, m, s_faulty):
    # Lentra's attack having a single faulty CRT signature + public infor
    t = (pow(s_faulty, e, n) - m) % n
    g = gcd(n, t)
    if 1 < g < n:
        return g
    return None

In [25]:
n, e, d, p, q, dp, dq, qinv = generate_rsa_key(1024)

# Choose a random message representative m < n
m = secrets.randbelow(n - 2) + 2


# ------------------ RSA-CRT SIGNING ------------------

# Compute a faulty RSA-CRT signature:
# a fault is injected in the mod-p branch, so:
#   s_faulty ≡ m^d (mod q)
#   s_faulty ≠ m^d (mod p)
s_faulty = rsa_crt_sign(m, p, q, dp, dq, qinv, fault_in="p")


# Ground truth (for comparison with recovered factors)
print("True factors:")
print("p =", p)
print("q =", q)


# Factor n using only:
#   - one faulty CRT signature
#   - the public key (n, e)
#   - the message m
# gcd(n, s_faulty^e - m) reveals one prime factor
factor2 = lenstra_recover_with_public(n, e, m, s_faulty)
if factor2:
    other2 = n // factor2
    print("\nRecovered factor using faulty signature + public key:")
    print("factor =", factor2)
    print("other  =", other2)

True factors:
p = 8109537618364611622061483507039820417009989986169406519317177856972133914017753835462750268915551978472552847818210383172170678170787912126221643718241431
q = 6877788941110763329966416307784855807559200384107247621038256846331789210987302062772824549955726338967270946012566385363201233941963741045129917611936039

Recovered factor using faulty signature + public key:
factor = 6877788941110763329966416307784855807559200384107247621038256846331789210987302062772824549955726338967270946012566385363201233941963741045129917611936039
other  = 8109537618364611622061483507039820417009989986169406519317177856972133914017753835462750268915551978472552847818210383172170678170787912126221643718241431
